# Notebook R3 -- clock-robustness experiment R: real photographs (TickTockVQA, CC BY 4.0 rows), all five models
Dataset: TickTockVQA (paper "It's Time to Get It Right", arXiv 2603.08011), Hugging Face mirror `monica-sekoyan/TickTockVQA-segmented`. **Evaluation only, no redistribution:** the photos are downloaded inside this session by `prep_ticktock_photos.py`, are never written to the repository or to a shared dataset, and are NOT included in the output zip (only `data.csv` and the provenance manifest are). Cite the paper and the dataset wherever results are reported.
Selection (fixed in the pre-registration): split `test`, per-row `license == "CC BY 4.0"`, 500 rows drawn with `RandomState(0)` from the rows sorted by `sample_id`, clock-crop image column; the licence is the mirror's per-row label (not independently verified for the COCO rows).
Models: Qwen-3B, InternVL3-2B, Qwen-7B, InternVL3-8B, Gemma-3-4B-it (bf16); `eval_behavior` (500) + `describe_check` (500), same prompts and caps as the synthetic runs (16 / 120), greedy, `CLOCK_REP_PENALTY=1.0` for the Qwen models.
Each step logs to `logs/`; the zip is refreshed after EVERY step; a failing step is recorded and the next one runs (except after a failed gate).
Pre-registered in `PREREG_MECHANISM.md` (ninth entry for R, written before any R data existed; tenth entry for the Gemma E3d run).

| step | what |
|---|---|
| 00 | **GATE** download the `test` shards (not `test_small`, not `image_full`), filter, sample 500, write PNGs + `data.csv` + provenance manifest. If it fails (no internet, schema change, fewer than 500 CC BY 4.0 rows), every later step is SKIPPED. |
| 01-05 | Qwen-3B, InternVL3-2B, Qwen-7B, InternVL3-8B, Gemma-4B: eval + describe each, cache purged between models |

## Inputs
| Dataset | Contents at the dataset ROOT |
|---|---|
| **clock-code** (UPDATED: `prep_ticktock_photos.py`) | `adapters.py`, `clocks.py`, `eval_behavior.py`, `describe_check.py`, `textnorm.py`, `gemma_diag.py`, `prep_ticktock_photos.py` |
No image dataset is needed (the photos come from Hugging Face; pyarrow and huggingface_hub are preinstalled on Kaggle).

**HF token:** Kaggle Secret `HF_TOKEN` (attached) for the gated Gemma repo only. **Settings:** GPU T4 x2, Internet ON, Save & Run All. **Output:** `mech_R3_results.zip`.
Runtime: a GUESS of about 5 h (range 4-9 h): measured standard-run step times (describe 24.7 / 13.8 / 33.2 / 21.1 / 80.1 min and eval 6.1 / 3.1 / 7.7 / 6.1 / 40.4 min for Qwen-3B / InternVL3-2B / Qwen-7B / InternVL3-8B / Gemma-4B) with a ~30% margin for photographs (resolution and token counts differ), plus ~15 min for the download and ~30 min of model loading. No step starts after 8 h.


In [ ]:
# CELL 1 -- GPU CHECK: stop the notebook right here if CUDA is not available
import torch, subprocess

MIN_GPUS = 2
if not torch.cuda.is_available():
    raise RuntimeError("STOP: CUDA is not available. Settings -> Accelerator -> 'GPU T4 x2', then Save & Run All again.")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
if n_gpu < MIN_GPUS:
    raise RuntimeError(f"STOP: this notebook needs {MIN_GPUS} GPU(s) but only {n_gpu} visible. Choose 'GPU T4 x2'.")
subprocess.run(["nvidia-smi"])
print("GPU check passed.")


In [ ]:
# CELL 2 -- COPY INPUTS from the attached Kaggle datasets into a clean working folder
import os, shutil, sys, time, json, subprocess, zipfile, datetime, hashlib

T_START = time.time()
INPUT, WORK = "/kaggle/input", "/kaggle/working"
RUN = f"{WORK}/run"
OUT_ZIP = f"{WORK}/mech_R3_results.zip"
CODE_FILES = ["adapters.py", "clocks.py", "eval_behavior.py", "describe_check.py", "text_only_v2.py", "conversion_test.py", "textnorm.py", "gemma_diag.py", "prep_ticktock_photos.py"]
EXPECT = {}

shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(f"{RUN}/logs")

def find_root(marker, maxdepth=4):
    """Directory under /kaggle/input that contains `marker` (a relative path). Dataset slugs don't matter."""
    base = INPUT.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(INPUT):
        if dirpath.count("/") - base > maxdepth:
            dirnames[:] = []
            continue
        if os.path.exists(os.path.join(dirpath, marker)):
            return dirpath
    raise FileNotFoundError(f"Could not find '{marker}' under {INPUT}. Attach the dataset that contains it "
                            f"(see the checklist at the top of this notebook).")

code_root = find_root("adapters.py")
for f in CODE_FILES:
    shutil.copy(os.path.join(code_root, f), RUN)
print("code from:", code_root)

for name, expect in EXPECT.items():
    root = find_root(f"{name}/data.csv")
    shutil.copytree(os.path.join(root, name), f"{RUN}/{name}")
    n_png = len([x for x in os.listdir(f"{RUN}/{name}") if x.endswith(".png")])
    sha = hashlib.sha256(open(f"{RUN}/{name}/data.csv", "rb").read()).hexdigest()[:12]
    print(f"{name}/: {n_png} PNGs (expected {expect}), data.csv sha256[:12]={sha}, from {root}")
    assert n_png == expect, f"{name}/ has {n_png} PNGs, expected {expect}: wrong dataset attached?"
print("inputs copied OK.")
du = shutil.disk_usage("/"); print(f"disk: {du.free / 2**30:.1f} GiB free of {du.total / 2**30:.1f} GiB")


In [ ]:
# CELL 3 -- ENVIRONMENT: env vars, versions, internet check (model weights are downloaded from Hugging Face)
os.environ.update({"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"})
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(m.__name__, m.__version__) for m in (torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL)];"
    "print('cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "a required package is missing -- see the error above"
open(f"{RUN}/env.txt", "w").write(r.stdout + "\nstarted " + datetime.datetime.utcnow().isoformat() + "Z\n")
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
    print("internet OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(f"STOP: huggingface.co not reachable ({e}). Settings -> Internet -> ON.")

# ---- Hugging Face token for the GATED Gemma repos (Kaggle Secret named HF_TOKEN) ----
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("STOP: no Kaggle secret 'HF_TOKEN' (Add-ons -> Secrets -> add label HF_TOKEN, value = your hf_... token, tick Attached). "
                       f"Detail: {type(e).__name__}: {e}")
from huggingface_hub import hf_hub_download
for repo in ("google/gemma-3-4b-it",):
    try:
        hf_hub_download(repo, "config.json", token=os.environ["HF_TOKEN"])
        print(f"access OK: {repo}")
    except Exception as e:
        raise RuntimeError(f"STOP: cannot read {repo} ({type(e).__name__}). Open https://huggingface.co/{repo} while logged in and click 'Acknowledge license'; "
                           "make sure the token is a Read token created by the SAME account. Detail: " + str(e)[:300])


In [ ]:
# CELL 4 -- THE STEPS: (name, command, guessed_minutes)
M3B, A3B = "Qwen/Qwen2.5-VL-3B-Instruct", "qwen2.5-vl-3b"
M2B, A2B = "OpenGVLab/InternVL3-2B-hf", "internvl3-2b"
M7B, A7B = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen2.5-vl-7b"
M8B, A8B = "OpenGVLab/InternVL3-8B-hf", "internvl3-8b"
DEADLINE_H = 8.0      # do not START a new step after this many hours
HARD_END_H = 9.0      # do not start a step whose guessed duration would end after this

def e3(model_id, adapter, parts, extra=()):
    return ["python", "mech_e3.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]

def purge_cache(*repos):
    code = ("import shutil,os,glob;"
            "[shutil.rmtree(p, ignore_errors=True) for r in %r for p in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--'+r))]" % (list(repos),))
    return ["python", "-c", code]

RPQ = ("--rep_penalty", "1.0")      # Qwen models only
def e2(model_id, adapter, parts, extra=()):
    return ["python", "mech_e2.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]
SMOKE = lambda mid, ad: ["python", "adapters.py", "--smoke", "--model_id", mid, "--adapter", ad, "--max_new_tokens", "32"]
GD = lambda mode, mid, ad, extra=(): ["python", "gemma_diag.py", "--mode", mode, "--model_id", mid, "--adapter", ad, *extra]
os.environ["CLOCK_REP_PENALTY"] = "1.0"          # HEADLINE decoding for the Qwen models; ignored by InternVL3 and Gemma
os.environ["CLOCK_DTYPE"] = "bfloat16"          # Gemma only (the other adapters ignore it)
GATE_STEPS = {'00_prep_photos'}
STEPS = [
    ("00_prep_photos", ["python", "prep_ticktock_photos.py", "--out", "data_photos_tt500"], 15),
    ("01_qwen2.5-vl-3b_photos_eval", ["python", "eval_behavior.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "Qwen/Qwen2.5-VL-3B-Instruct", "--adapter", "qwen2.5-vl-3b", "--out_csv", "results/qwen2.5-vl-3b_photos/results.csv"], 9),
    ("01_qwen2.5-vl-3b_photos_describe", ["python", "describe_check.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "Qwen/Qwen2.5-VL-3B-Instruct", "--adapter", "qwen2.5-vl-3b", "--out_csv", "results/qwen2.5-vl-3b_photos/describe.csv"], 32),
    ("01_purge_qwen3b_cache", purge_cache("Qwen--Qwen2.5-VL-3B-Instruct"), 1),
    ("02_internvl3-2b_photos_eval", ["python", "eval_behavior.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "OpenGVLab/InternVL3-2B-hf", "--adapter", "internvl3-2b", "--out_csv", "results/internvl3-2b_photos/results.csv"], 5),
    ("02_internvl3-2b_photos_describe", ["python", "describe_check.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "OpenGVLab/InternVL3-2B-hf", "--adapter", "internvl3-2b", "--out_csv", "results/internvl3-2b_photos/describe.csv"], 18),
    ("02_purge_internvl2b_cache", purge_cache("OpenGVLab--InternVL3-2B-hf"), 1),
    ("03_qwen2.5-vl-7b_photos_eval", ["python", "eval_behavior.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "Qwen/Qwen2.5-VL-7B-Instruct", "--adapter", "qwen2.5-vl-7b", "--out_csv", "results/qwen2.5-vl-7b_photos/results.csv"], 11),
    ("03_qwen2.5-vl-7b_photos_describe", ["python", "describe_check.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "Qwen/Qwen2.5-VL-7B-Instruct", "--adapter", "qwen2.5-vl-7b", "--out_csv", "results/qwen2.5-vl-7b_photos/describe.csv"], 42),
    ("03_purge_qwen7b_cache", purge_cache("Qwen--Qwen2.5-VL-7B-Instruct"), 1),
    ("04_internvl3-8b_photos_eval", ["python", "eval_behavior.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "OpenGVLab/InternVL3-8B-hf", "--adapter", "internvl3-8b", "--out_csv", "results/internvl3-8b_photos/results.csv"], 8),
    ("04_internvl3-8b_photos_describe", ["python", "describe_check.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "OpenGVLab/InternVL3-8B-hf", "--adapter", "internvl3-8b", "--out_csv", "results/internvl3-8b_photos/describe.csv"], 28),
    ("04_purge_internvl8b_cache", purge_cache("OpenGVLab--InternVL3-8B-hf"), 1),
    ("05_gemma-3-4b-it_photos_eval", ["python", "eval_behavior.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "google/gemma-3-4b-it", "--adapter", "gemma-3-4b-it", "--out_csv", "results/gemma-3-4b-it_photos/results.csv"], 45),
    ("05_gemma-3-4b-it_photos_describe", ["python", "describe_check.py", "--data_csv", "data_photos_tt500/data.csv", "--images_dir", "data_photos_tt500", "--model_id", "google/gemma-3-4b-it", "--adapter", "gemma-3-4b-it", "--out_csv", "results/gemma-3-4b-it_photos/describe.csv"], 85),
    ("05_purge_gemma4b_cache", purge_cache("google--gemma-3-4b-it"), 1),
]
for name, cmd, est in STEPS:
    print(f"{name:<32} ~{est:>3} min (guess)  {' '.join(cmd)[:110]}")
print(f"guessed total: {sum(e for _, _, e in STEPS) / 60:.1f} h")


In [ ]:
# CELL 5 -- RUN every step. A failing step is recorded and the notebook CONTINUES with the next one.
EXPECTED = ['results/qwen2.5-vl-3b_photos/results.csv', 'results/qwen2.5-vl-3b_photos/describe.csv', 'results/internvl3-2b_photos/results.csv', 'results/internvl3-2b_photos/describe.csv', 'results/qwen2.5-vl-7b_photos/results.csv', 'results/qwen2.5-vl-7b_photos/describe.csv', 'results/internvl3-8b_photos/results.csv', 'results/internvl3-8b_photos/describe.csv', 'results/gemma-3-4b-it_photos/results.csv', 'results/gemma-3-4b-it_photos/describe.csv', 'data_photos_tt500/manifest_provenance.csv']        # files that should exist at the end (relative to the run folder)
status = []

def make_zip():
    """Whitelisted snapshot of everything worth keeping; refreshed after EVERY step."""
    tmp = OUT_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs"):
            for dp, _, fns in os.walk(f"{RUN}/{sub}"):
                for fn in fns:
                    full = os.path.join(dp, fn); z.write(full, os.path.relpath(full, RUN))
        for rel in ["status.json", "STATUS.txt", "env.txt", "conversion_images/manifest.csv", "data_photos_tt500/data.csv", "data_photos_tt500/manifest_provenance.csv"] + CODE_FILES:
            if os.path.exists(f"{RUN}/{rel}"):
                z.write(f"{RUN}/{rel}", rel)
    os.replace(tmp, OUT_ZIP)

def run_step(name, cmd, est_min):
    elapsed_h = (time.time() - T_START) / 3600
    if elapsed_h > DEADLINE_H or elapsed_h + est_min / 60 > HARD_END_H:
        return {"step": name, "status": "SKIPPED (time budget)", "returncode": None, "minutes": 0.0}
    cmd = [sys.executable if c == "python" else c for c in cmd]
    print(f"\n{'=' * 100}\n[{datetime.datetime.utcnow():%H:%M:%S}] START {name}  (elapsed {elapsed_h:.2f} h)\n{'=' * 100}", flush=True)
    t0 = time.time()
    with open(f"{RUN}/logs/{name}.log", "w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = p.wait()
    mins = (time.time() - t0) / 60
    print(f"[{name}] finished rc={rc} in {mins:.1f} min", flush=True)
    return {"step": name, "status": "OK" if rc == 0 else f"FAILED rc={rc}", "returncode": rc, "minutes": round(mins, 1)}

gate_failed = False
for name, cmd, est in STEPS:
    if gate_failed:
        status.append({"step": name, "status": "SKIPPED (gate failed)", "returncode": None, "minutes": 0.0})
        json.dump(status, open(f"{RUN}/status.json", "w"), indent=1); make_zip(); continue
    try:
        status.append(run_step(name, cmd, est))
    except Exception as e:           # never let one step's crash stop the notebook
        status.append({"step": name, "status": f"ERROR {type(e).__name__}: {e}"[:120], "returncode": None, "minutes": 0.0})
        print(f"[{name}] notebook-side error: {e}", flush=True)
    if name in GATE_STEPS and status[-1]["status"] != "OK":
        gate_failed = True
        print(f"GATE {name} FAILED -> all later steps are skipped; read logs/{name}.log", flush=True)
    json.dump(status, open(f"{RUN}/status.json", "w"), indent=1)
    make_zip()                       # insurance: the zip is valid after every step
    print(f"zip refreshed after {name}: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB", flush=True)


In [ ]:
# CELL 6 -- FINAL: status table, completeness check, ONE output zip
lines = [f"{s['step']:<32} {s['status']:<26} {s['minutes']:>7} min" for s in status]
missing = [f for f in EXPECTED if not os.path.exists(f"{RUN}/{f}")]
lines += ["", f"total wall time: {(time.time() - T_START) / 3600:.2f} h",
          "MISSING expected files: " + (", ".join(missing) if missing else "none")]
open(f"{RUN}/STATUS.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
make_zip()
with zipfile.ZipFile(OUT_ZIP) as z:
    assert z.testzip() is None, "zip is corrupt"
    names = z.namelist()
print(f"\nOUTPUT: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB, {len(names)} files)")
for n in sorted(names):
    if n.startswith("results/") and n.endswith((".csv", ".json", ".png")):
        print("  ", n)
shutil.rmtree(RUN)      # leave ONLY the zip in /kaggle/working (the zip is verified above)
print("\nDONE." if not missing and all(s["status"] == "OK" for s in status) else
      "\nDONE WITH PROBLEMS -- read STATUS.txt inside the zip and the logs/ folder.")
